[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)

# Lesson 14 — Regular Expressions

**Module 1 — Python Fundamentals** | ⏱ 25 min

Regular expressions (regex) are a mini-language for describing text patterns. They let you search, extract, validate, and transform text in ways that would require dozens of lines of manual string parsing. Python's `re` module provides full regex support. While the syntax looks cryptic at first, a small set of patterns covers the vast majority of real-world use cases.

## Learning Objectives
- Use `re.search()`, `re.match()`, `re.findall()`, `re.finditer()`, `re.sub()`, and `re.split()`
- Build patterns using character classes `[\w\d\s]`, quantifiers `*+?{n,m}`, and anchors `^$`
- Use groups `()` to capture and extract specific parts of a match
- Use named groups `(?P<name>...)` for readable extractions
- Compile patterns with `re.compile()` for performance
- Apply regex to real-world tasks: email validation, log parsing, URL extraction

## Core re Functions

The `re` module provides several functions, each serving a different purpose. `re.search()` scans through a string looking for the first location where the pattern produces a match. `re.match()` only checks at the **beginning** of the string. `re.findall()` returns a list of all non-overlapping matches. `re.finditer()` returns an iterator of match objects (more information than `findall()`). Always use **raw strings** (prefix `r`) for regex patterns to avoid backslash issues.

In [ ]:
import re

# re.search() — find first match anywhere in the string
text = "Order #4521 placed on 2024-01-15 for $149.99"

order_match = re.search(r"#(\d+)", text)         # Find order number
date_match  = re.search(r"\d{4}-\d{2}-\d{2}", text) # Find date
price_match = re.search(r"\$(\d+\.\d{2})", text)  # Find price

if order_match:
    print(f"Order number: {order_match.group(1)}")  # group(1) = first capture group
if date_match:
    print(f"Date:         {date_match.group()}")    # group(0) or group() = full match
if price_match:
    print(f"Price:        {price_match.group(1)}")

# re.match() — only matches at the START of the string
print()
print(bool(re.match(r"Order", text)))   # True  — 'Order' is at start
print(bool(re.match(r"\d{4}", text)))   # False — string starts with 'Order', not digits
print(bool(re.search(r"\d{4}", text)))  # True  — search finds it anywhere

In [ ]:
import re

# re.findall() — return all matches as a list
log = """
192.168.1.1 - alice [15/Jan/2024] GET /api/users 200
10.0.0.5 - bob [15/Jan/2024] POST /api/orders 201
192.168.1.100 - carol [15/Jan/2024] GET /api/products 200
10.0.0.8 - admin [15/Jan/2024] DELETE /api/items/42 404
"""

# Find all IP addresses
ip_pattern = r"\b(?:\d{1,3}\.){3}\d{1,3}\b"
ip_addresses = re.findall(ip_pattern, log)
print(f"IP addresses: {ip_addresses}")

# Find all HTTP methods
http_methods = re.findall(r"\b(GET|POST|PUT|DELETE|PATCH)\b", log)
print(f"HTTP methods: {http_methods}")

# Find all status codes
status_codes = re.findall(r"\b(200|201|400|404|500)\b", log)
print(f"Status codes: {status_codes}")

# re.finditer() — returns match objects with position info
print("\nPath endpoints:")
for match in re.finditer(r"/api/[\w/]+", log):
    print(f"  Found '{match.group()}' at position {match.start()}-{match.end()}")

## Pattern Building Blocks

Regex patterns are built from a set of special symbols. **Character classes** match types of characters: `\d` matches digits, `\w` matches word characters (letters, digits, underscore), `\s` matches whitespace, and `[abc]` matches any of a, b, or c. **Quantifiers** control repetition: `*` (0 or more), `+` (1 or more), `?` (0 or 1), `{n}` (exactly n), `{n,m}` (between n and m). **Anchors** `^` and `$` match position (start and end of string/line).

In [ ]:
import re

# Character classes
test = "Hello World 123 foo_bar@example.com 2024-01-15"

print("Character class matches:")
print(f"  \\d  (digits):     {re.findall(r'\d', test)}")        # Single digits
print(f"  \\d+ (numbers):   {re.findall(r'\d+', test)}")       # Sequences of digits
print(f"  \\w+ (words):     {re.findall(r'\w+', test)}")       # Word characters
print(f"  \\s+ (spaces):    {[repr(s) for s in re.findall(r'\s+', test)]}") # Whitespace
print(f"  [A-Z] (capitals): {re.findall(r'[A-Z]', test)}")     # Uppercase only
print(f"  [^\\d] (non-dig):  {''.join(re.findall(r'[^\d]', test))}")  # NOT digits

# Quantifiers
patterns = [
    (r"\d{3}",    "Exactly 3 digits"),
    (r"\d{2,4}",  "2 to 4 digits"),
    (r"\w+",      "One or more word chars"),
    (r"\d*\.\d+", "Optional digits, dot, digits (decimal)"),
    (r"colou?r",  "'color' or 'colour' (u is optional)"),
]
test2 = "color 123 colour 1234 45.6 7890"
print("\nQuantifier examples on:", repr(test2))
for pattern, desc in patterns:
    matches = re.findall(pattern, test2)
    print(f"  {pattern:<16} ({desc}): {matches}")

In [ ]:
import re

# Anchors: ^ (start) and $ (end)
lines = [
    "ERROR: disk full",
    "WARNING: low memory",
    "INFO: process started",
    "This is an ERROR too",
    "DEBUG: query time 200ms",
]

# ^ anchor — line must START with ERROR
start_errors = [l for l in lines if re.match(r"^ERROR", l)]
print("Lines starting with ERROR:")
for line in start_errors:
    print(f"  {line}")

# $ anchor — line must END with a number
with_numbers = [l for l in lines if re.search(r"\d+$", l)]
print("\nLines ending with a number:")
for line in with_numbers:
    print(f"  {line}")

# re.MULTILINE flag — ^ and $ match at each line start/end in multi-line text
multiline_text = """apple\nbanana\navocado\nberry"""
starts_with_a = re.findall(r"^a\w+", multiline_text, re.MULTILINE)
print(f"\nWords starting with 'a': {starts_with_a}")

# re.IGNORECASE flag
print(f"\nCase-insensitive search for 'error':")
for line in lines:
    if re.search(r"error", line, re.IGNORECASE):
        print(f"  {line}")

## Groups and Capture

Parentheses `()` in a regex create a **capture group** — the substring matched by that part of the pattern is captured separately. `match.group(0)` returns the entire match, `match.group(1)` returns the first group, `match.group(2)` the second, and so on. **Named groups** `(?P<name>...)` let you access captures by name rather than index, making complex patterns much more readable and maintainable.

In [ ]:
import re

# Basic groups — numbered captures
log_line = "2024-01-15 14:32:07 ERROR [auth_service] User 'alice' login failed (3 attempts)"

# Group 1: date, Group 2: time, Group 3: level, Group 4: service
pattern = r"(\d{4}-\d{2}-\d{2}) (\d{2}:\d{2}:\d{2}) (\w+) \[(\w+)\]"
match = re.search(pattern, log_line)
if match:
    print(f"Full match: {match.group(0)}")
    print(f"Date:       {match.group(1)}")
    print(f"Time:       {match.group(2)}")
    print(f"Level:      {match.group(3)}")
    print(f"Service:    {match.group(4)}")
    # All groups as a tuple
    print(f"All groups: {match.groups()}")

In [ ]:
import re

# Named groups — ?P<name> makes patterns self-documenting
log_line = "2024-01-15 14:32:07 ERROR [auth_service] User 'alice' login failed (3 attempts)"

pattern = r"""(?P<date>\d{4}-\d{2}-\d{2})\s+
              (?P<time>\d{2}:\d{2}:\d{2})\s+
              (?P<level>\w+)\s+
              \[(?P<service>\w+)\]"""

match = re.search(pattern, log_line, re.VERBOSE)  # re.VERBOSE allows whitespace/comments
if match:
    d = match.groupdict()  # Returns a dict of all named groups
    print(f"Date:    {d['date']}")
    print(f"Time:    {d['time']}")
    print(f"Level:   {d['level']}")
    print(f"Service: {d['service']}")

# Parse multiple log lines using finditer with named groups
log_data = """
2024-01-15 10:00:01 INFO  [web_server] Started on port 8080
2024-01-15 10:05:33 ERROR [database] Connection timeout after 30s
2024-01-15 10:05:34 WARN  [database] Retrying connection (1/3)
2024-01-15 10:06:01 ERROR [database] Connection timeout after 30s
2024-01-15 10:06:02 INFO  [database] Connected to replica
"""

entry_pattern = re.compile(
    r"(?P<date>\d{4}-\d{2}-\d{2}) (?P<time>\d{2}:\d{2}:\d{2}) "
    r"(?P<level>\w+)\s+\[(?P<service>\w+)\] (?P<message>.+)"
)

print("\nParsed log entries:")
errors = []
for match in entry_pattern.finditer(log_data):
    d = match.groupdict()
    print(f"  [{d['level']:<5}] {d['service']}: {d['message'][:40]}")
    if d['level'] == 'ERROR':
        errors.append(d)

print(f"\n{len(errors)} error(s) found:")
for e in errors:
    print(f"  {e['time']} - {e['service']}: {e['message']}")

## re.sub() and re.split()

`re.sub()` finds all matches of a pattern and replaces them with a replacement string or the result of a function. The replacement can include backreferences to captured groups using `\1`, `\2`, or `\g<name>`. `re.split()` splits a string on a pattern rather than a literal string, which is much more flexible than `str.split()`.

In [ ]:
import re

# re.sub() — find and replace with regex

# 1. Simple replacement
text = "The year 2024 was great! 2025 will be even better."
new_text = re.sub(r"\d{4}", "YEAR", text)
print(new_text)  # "The year YEAR was great! YEAR will be even better."

# 2. Backreferences — rearrange captured groups
date_str = "2024-01-15"  # YYYY-MM-DD
reformatted = re.sub(r"(\d{4})-(\d{2})-(\d{2})", r"\3/\2/\1", date_str)
print(reformatted)  # 15/01/2024  (rearranged to DD/MM/YYYY)

# Using named groups in substitution
reformatted2 = re.sub(
    r"(?P<y>\d{4})-(?P<m>\d{2})-(?P<d>\d{2})",
    r"\g<d>/\g<m>/\g<y>",
    date_str
)
print(reformatted2)  # 15/01/2024

# 3. Replacement with a function (for dynamic replacements)
def mask_card(match):
    """Mask all but last 4 digits of a credit card number."""
    digits = match.group()
    return "*" * (len(digits) - 4) + digits[-4:]

payment_log = "Card 4532015112830366 charged $99.99 on card 5425233430109903"
masked = re.sub(r"\b\d{16}\b", mask_card, payment_log)
print(f"\nMasked: {masked}")

# re.split() — split on a pattern
mixed_text = "one  two   three,four;five:six"
# Split on one or more spaces, commas, semicolons, or colons
parts = re.split(r"[\s,;:]+", mixed_text)
print(f"\nSplit result: {parts}")

## re.compile() for Performance

When you use the same pattern many times, compiling it once with `re.compile()` is more efficient because the pattern only needs to be parsed and converted to bytecode once. The compiled pattern object has all the same methods (`search`, `findall`, `sub`, etc.) as the `re` module. This is especially important in loops processing thousands of records.

In [ ]:
import re
import time

# Compile a pattern once for repeated use
email_pattern = re.compile(
    r"[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,}"
)

# Use the compiled pattern (same API as re module functions)
test_emails = [
    "user@example.com",
    "not.an.email",
    "name+tag@domain.co.uk",
    "UPPER@CASE.COM",
    "missing@.com",
    "admin_2024@company.org",
    "two@@signs.com",
]

print("Email validation:")
for email in test_emails:
    match = email_pattern.fullmatch(email)  # fullmatch: entire string must match
    status = "valid" if match else "invalid"
    print(f"  {email:<30} -> {status}")

# Performance demonstration
n_iterations = 100_000
test_string = "Contact us at support@company.com or sales@company.co.uk"

start = time.time()
for _ in range(n_iterations):
    re.findall(r"[\w.+\-]+@[\w.\-]+\.\w+", test_string)
uncompiled_time = time.time() - start

pattern = re.compile(r"[\w.+\-]+@[\w.\-]+\.\w+")
start = time.time()
for _ in range(n_iterations):
    pattern.findall(test_string)
compiled_time = time.time() - start

print(f"\n{n_iterations:,} iterations:")
print(f"  Without compile: {uncompiled_time:.3f}s")
print(f"  With compile:    {compiled_time:.3f}s")
print(f"  Speedup:         {uncompiled_time/compiled_time:.1f}x")

## Real-World Parsing: Email, URLs, and Logs

Regular expressions shine on practical parsing tasks. Here we solve three common real-world problems: extracting and validating emails, finding all URLs in a document, and parsing structured log files into Python dictionaries.

In [ ]:
import re

# Real-world task 1: Extract all URLs from a web page excerpt
html_content = """
<html>
  <a href="https://www.python.org">Python official site</a>
  <a href="http://docs.python.org/3/library/re.html">re module docs</a>
  <img src="https://cdn.example.com/images/logo.png" alt="Logo">
  <a href="https://github.com/python/cpython">CPython GitHub</a>
  <a href="/relative/path">Relative link (not a URL)</a>
  Visit https://pypi.org for packages and http://legacy.pypi.org for the old site.
"""

url_pattern = re.compile(r"https?://[^\s\"'<>]+'")
urls = re.compile(r"https?://[^\s\"'<>]+").findall(html_content)

print(f"Found {len(urls)} absolute URLs:")
for url in urls:
    scheme = "HTTPS" if url.startswith("https") else "HTTP "
    print(f"  [{scheme}] {url}")

https_count = sum(1 for u in urls if u.startswith("https"))
http_count  = len(urls) - https_count
print(f"\nHTTPS: {https_count}, HTTP: {http_count}")

In [ ]:
import re

# Real-world task 2: Parse Apache-style access logs into structured data
access_log = """
192.168.1.10 - alice [15/Jan/2024:10:00:01 +0000] "GET /api/users HTTP/1.1" 200 1523
10.0.0.5 - bob [15/Jan/2024:10:01:33 +0000] "POST /api/orders HTTP/1.1" 201 256
192.168.1.15 - - [15/Jan/2024:10:02:05 +0000] "GET /favicon.ico HTTP/1.1" 404 0
10.0.0.8 - admin [15/Jan/2024:10:03:21 +0000] "DELETE /api/items/42 HTTP/1.1" 200 45
192.168.1.99 - eve [15/Jan/2024:10:05:00 +0000] "GET /api/admin HTTP/1.1" 403 89
"""

log_pattern = re.compile(
    r'(?P<ip>[\d.]+) - (?P<user>[\w-]+) '
    r'\[(?P<datetime>[^\]]+)\] '
    r'"(?P<method>\w+) (?P<path>[^\s]+) HTTP/[\d.]+" '
    r'(?P<status>\d{3}) (?P<bytes>\d+)'
)

print(f"{'IP':<16} {'User':<8} {'Method':<7} {'Path':<20} {'Status'} {'Bytes'}")
print("-" * 70)

parsed_entries = []
for match in log_pattern.finditer(access_log):
    d = match.groupdict()
    parsed_entries.append(d)
    print(f"{d['ip']:<16} {d['user']:<8} {d['method']:<7} {d['path']:<20} {d['status']}   {d['bytes']:>6}")

# Analyse: count by status code
from collections import Counter
status_counts = Counter(e['status'] for e in parsed_entries)
print(f"\nStatus code breakdown: {dict(status_counts)}")
print(f"Total bytes transferred: {sum(int(e['bytes']) for e in parsed_entries):,}")

## Practice Exercises

1. Write a function `validate_password(password)` using regex that returns a list of all failed requirements. Requirements: at least 8 characters, at least one uppercase letter, at least one lowercase letter, at least one digit, and at least one special character from `!@#$%^&*`.
2. Write a regex to extract all phone numbers from a block of text. Phone numbers can appear in formats like `(416) 555-1234`, `416-555-1234`, `416.555.1234`, `+1 416 555 1234`, or `4165551234`. Group the area code, exchange, and number separately.
3. Parse a configuration file where each non-comment line has the format `KEY = VALUE` (with optional spaces around `=`). Comments start with `#`. Use named groups and `re.finditer()` to build a dictionary of all settings.
4. Write a function `highlight_keywords(text, keywords)` that uses `re.sub()` with a compiled pattern to wrap any occurrence of a keyword (case-insensitive) in `**...**` markdown bold markers. Test with a paragraph of text and at least 4 keywords.